In [9]:
# --- Step 1: Install & Import ---
!pip install -q gspread
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default
from google.colab import drive
import os # We now need the 'os' module for file system operations

# --- Step 2: Authentication & Mounting Drive ---
print("Authenticating user for Google services...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("✅ G-Sheets access authorized.")

print("\nMounting Google Drive...")
# This will trigger an authorization pop-up for Drive access.
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully at '/content/drive'.")


# --- Step 3: CRITICAL CONFIGURATION ---
# === YOU MUST UPDATE THESE VALUES ===

# A. The base path to the directory containing all your session screenshot folders.
#    NOTE: It always starts with '/content/drive/My Drive/'.
DRIVE_SCREENSHOTS_ROOT_PATH = '/content/drive/MyDrive/Opus_1_101001/Media/Screenshots' # <-- e.g., '/content/drive/My Drive/Opus1_101001/Media/Screenshots'

# B. The map linking Session IDs from your spreadsheet to their folder names.
#    KEY = Session_ID (string) | VALUE = Folder Name (string)
SESSION_TO_FOLDER_MAP = {
    'SID0001': 'Screenshots_1',
    'SID0002': 'Screenshots_2',
    'SID0003': 'Screenshots_3',
    'SID0004': 'Screenshots_4',
    'SID0005-1': 'Screenshots_5', # As per your special instruction
    'SID0005-2': 'Screenshots_5', # As per your special instruction
    'SID0006': 'Screenshots_6',
    'SID0007': 'Screenshots_7',
    'SID0008': 'Screenshots_8',
    'SID0009': 'Screenshots_9',
    'SID0010': 'Screenshots_10',
    'SID0011': 'Screenshots_11',
    'SID0012': 'Screenshots_12',
    'SID0013': 'Screenshots_13',
    'SID0014': 'Screenshots_14',
    'SID0015': 'Screenshots_15',
    'SID0016': 'Screenshots_16',
    'SID0017': 'Screenshots_17',
    'SID0018': 'Screenshots_18',
    'SID0019': 'Screenshots_19',
    'SID0020': 'Screenshots_20',
    'SID0021': 'Screenshots_21',
    'SID0022': 'Screenshots_22',
    'SID0023': 'Screenshots_23',
    'SID0024': 'Screenshots_24',
    'SID0025': 'Screenshots_25',
    'SID0026': 'Screenshots_26',
    'SID0027': 'Screenshots_27',
    'SID0028': 'Screenshots_28',
    'SID0029': 'Screenshots_29',
    'SID0030': 'Screenshots_30',
    'SID0031': 'Screenshots_31',
    'SID0032': 'Screenshots_32',
    'SID0033': 'Screenshots_33',
    'SID0034': 'Screenshots_34',
    'SID0035': 'Screenshots_35',
    'SID0036': 'Screenshots_36',
    'SID0037': 'Screenshots_37',
    'SID0038': 'Screenshots_38',
    'SID0039': 'Screenshots_39',
    'SID0040': 'Screenshots_40',
    'SID0041': 'Screenshots_41',
    'SID0042': 'Screenshots_42',
    'SID0043': 'Screenshots_43',
    'SID0044': 'Screenshots_44',
    'SID0045': 'Screenshots_45',
    'SID0046': 'Screenshots_46',
    'SID0047': 'Screenshots_47',
    'SID0048': 'Screenshots_48',
    'SID0049': 'Screenshots_49',
    'SID0050': 'Screenshots_50',
    'SID0051': 'Screenshots_51',
    'SID0052': 'Screenshots_52',
    'SID0053': 'Screenshots_53',
    'SID0054': 'Screenshots_54',
    'SID0055': 'Screenshots_55',
    'SID0056': 'Screenshots_56',
    'SID0057': 'Screenshots_57',
    'SID0058': 'Screenshots_58',
    'SID0059': 'Screenshots_59',
    'SID0060': 'Screenshots_60',
    'SID0061': 'Screenshots_61',
    'SID0062': 'Screenshots_62',
    'SID0063': 'Screenshots_63',
    'SID0064': 'Screenshots_64',
    'SID0065': 'Screenshots_65',
    'SID0066': 'Screenshots_66',
    'SID0067': 'Screenshots_67',
    'SID0068': 'Screenshots_68',
}

# C. Spreadsheet Details (as before)
SPREADSHEET_NAME = 'RAW_Requests'
WORKSHEET_NAME = 'raw_requests_polished'
SESSION_ID_COLUMN = 'Session_ID'
IMAGE_COLUMN_NAME = 'Image'

# --- MAIN EXECUTION ---
print("\n--- `Automated Master Session Audit` ---")
try:
    # --- Phase 1: Read Live Data from Google Sheets ---
    print("Reading session log from Google Sheets...")
    worksheet = gc.open(SPREADSHEET_NAME).worksheet(WORKSHEET_NAME)
    df = pd.DataFrame(worksheet.get_all_records())

    image_df = df[df[IMAGE_COLUMN_NAME].astype(str).str.startswith('IMG', na=False)]
    session_image_counts = image_df[SESSION_ID_COLUMN].value_counts()

    summary_df = session_image_counts.reset_index()
    summary_df.columns = ['Session_ID', 'LogImageCount']
    summary_df = summary_df.sort_values(by='Session_ID').reset_index(drop=True)

    # --- Phase 2: Automated File System Count from Google Drive ---
    print("Beginning automated audit of Google Drive folders...")
    filesystem_counts = []

    for session_id in summary_df['Session_ID']:
        folder_name = SESSION_TO_FOLDER_MAP.get(session_id)

        if folder_name is None:
            print(f"   ⚠️ WARNING: No folder mapping found for session '{session_id}'. Count set to 0.")
            filesystem_counts.append(0)
            continue

        full_path = os.path.join(DRIVE_SCREENSHOTS_ROOT_PATH, folder_name)

        if not os.path.isdir(full_path):
            print(f"   ❌ ERROR: Folder not found at '{full_path}' for session '{session_id}'. Count set to 0.")
            filesystem_counts.append(0)
        else:
            # Count only .png files to avoid counting other system files
            png_files = [f for f in os.listdir(full_path) if f.lower().endswith('.png')]
            count = len(png_files)
            filesystem_counts.append(count)
            print(f"   ✅ Session '{session_id}' -> Folder '{folder_name}': Found {count} images.")

    # --- Phase 3: Create and Save Final Report ---
    print("\n--- Audit Complete ---")
    summary_df['FileSystemImageCount'] = filesystem_counts
    summary_df['Discrepancy'] = summary_df['FileSystemImageCount'] - summary_df['LogImageCount']

    print("\n--- Final Discrepancy Report ---")
    print(summary_df.to_string())

    output_filename = 'automated_discrepancy_report.csv'
    summary_df.to_csv(output_filename, index=False)
    print(f"\n✅ Report successfully saved as '{output_filename}'.")

except Exception as e:
    print(f"❌ An unrecoverable error occurred: {e}")

Authenticating user for Google services...
✅ G-Sheets access authorized.

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully at '/content/drive'.

--- `Automated Master Session Audit` ---
Reading session log from Google Sheets...
Beginning automated audit of Google Drive folders...
   ✅ Session 'SID0001' -> Folder 'Screenshots_1': Found 34 images.
   ✅ Session 'SID0002' -> Folder 'Screenshots_2': Found 55 images.
   ✅ Session 'SID0003' -> Folder 'Screenshots_3': Found 93 images.
   ✅ Session 'SID0004' -> Folder 'Screenshots_4': Found 88 images.
   ✅ Session 'SID0005-1' -> Folder 'Screenshots_5': Found 52 images.
   ✅ Session 'SID0005-2' -> Folder 'Screenshots_5': Found 52 images.
   ✅ Session 'SID0006' -> Folder 'Screenshots_6': Found 24 images.
   ✅ Session 'SID0007' -> Folder 'Screenshots_7': Found 51 images.
   ✅ Session 'SID0008' -> Folder 'Scre